# **LoRA Finetune method**

### Dependency
- *torch*
- *dataset*
- *transformer*
- *peft*
- *bitsandbits*

### Core deep learning
- torch>=2.1.0

### Hugging Face core (stable training stack)
- transformers>=4.41.0,<4.47.0
- accelerate>=0.31.0,<1.0.0
- datasets>=2.18.0
- huggingface-hub>=0.21.0

### PEFT / LoRA / QLoRA
- peft>=0.17.0,<0.19.0
- bitsandbytes>=0.43.0

### Utilities (often implicitly required)
- numpy>=1.24.0
- scipy
- tqdm
- packaging
- sentencepiece
- protobuf

## **When doing LoRA / QLoRA:**

- Transformers 4.46.3

- Accelerate 0.34.x

- PEFT 0.17–0.18


In [1]:
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)

from peft import LoraConfig, get_peft_model, TaskType


In [2]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)


In [3]:
lora_config = LoraConfig(
    r = 8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj"],
    lora_dropout=0.05,
    bias='none',\
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

In [4]:
data = load_dataset("openai/gsm8k", "main", split="train[:1000]")

In [5]:
def tokenize(batch):
    texts = [
        f"### Instruction:\n{instruction}\n### Response:\n{out}"
        for instruction, out in zip(batch["question"], batch["answer"])
    ]

    tokens = tokenizer(
        texts,
        padding="max_length",
        max_length=256,
        truncation=True,
        return_tensors="pt"
    )

    tokens["labels"] = tokens["input_ids"].clone()

    return tokens


In [6]:
tokenized_data = data.map(tokenize, batched=True, remove_columns=data.column_names)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir="./tinylama-math-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-3,
    num_train_epochs=50,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"]
)


/home/md-al-amin/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/md-al-amin/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/home/md-al-amin/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [8]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data,
    tokenizer=tokenizer
)


/tmp/ipykernel_141538/2970692923.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/md-al-amin/miniconda3/envs/torch-gpu/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [9]:
trainer.train()

Step,Training Loss
20,2.115100
40,0.846000
60,0.814100
80,0.727600
100,0.752500
120,0.749600
140,0.718200
160,0.644900
180,0.709800
200,0.659700


TrainOutput(global_step=3100, training_loss=0.20345881000641852, metrics={'train_runtime': 4926.4083, 'train_samples_per_second': 10.149, 'train_steps_per_second': 0.629, 'total_flos': 7.89316557078528e+16, 'train_loss': 0.20345881000641852, 'epoch': 49.6})

## **Model Save to the `Local`**

In [10]:
model.save_pretrained("./tinylama-math-lora-tuned-adapter")
tokenizer.save_pretrained("./tinylama-math-lora-tuned-adapter")

('./tinylama-math-lora-tuned-adapter/tokenizer_config.json',
 './tinylama-math-lora-tuned-adapter/special_tokens_map.json',
 './tinylama-math-lora-tuned-adapter/tokenizer.json')